# 12 — Capstone: Integrated Security Operations Platform

## What This Is
This capstone integrates the techniques from all prior notebooks into a simulated Security Operations Centre (SOC) pipeline: static code scanning → network recon → phishing detection → anomaly detection → threat intelligence → red team coverage gap analysis.

## Why It Matters
Real SOCs face alert fatigue from siloed tools. Integrating signals — correlating a phishing email with an anomalous network flow and a TTP match — dramatically improves detection confidence and reduces false positives.

## Where the Community Stands
SIEM platforms (Splunk, Microsoft Sentinel, Google SecOps) serve as integration hubs. SOAR (Security Orchestration, Automation, Response) automates the workflow between tools. XDR (Extended Detection & Response) provides unified telemetry across endpoint, network, and identity.

## What You've Built Across This Series
Across 12 notebooks you implemented from first principles: static vulnerability analysis, fuzzing/mutation, network recon scoring, credential stuffing detection, phishing NLP, web app DAST, malware static analysis + YARA, threat intelligence attribution, red team operation management, and ML-based anomaly detection. These are the building blocks of a modern security programme.

In [ ]:
# Integrated SOC pipeline: pulls together utilities from across the series
import re, hashlib, math, random, statistics
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from urllib.parse import urlparse

# ---- Reuse components from prior notebooks (inline for standalone execution) ----

URGENT_WORDS = [
    'urgent', 'action required', 'verify now', 'suspended',
    'confirm your', 'click here', 'expires', 'your account',
]
SUSPICIOUS_TLDS = {'.tk', '.ml', '.ga', '.cf', '.gq', '.xyz', '.top'}
TRUSTED_DOMAINS = {'gmail.com','microsoft.com','apple.com','paypal.com'}

def phishing_score_fast(email: Dict) -> float:
    subject = email.get('subject','').lower()
    body    = email.get('body','').lower()
    sender  = email.get('from','').lower()
    urls    = email.get('urls', [])
    text    = subject + ' ' + body
    score   = 0.0
    urgency = sum(1 for w in URGENT_WORDS if w in text)
    score  += min(0.30, urgency * 0.08)
    m = re.search(r'@([\w\.-]+)', sender)
    if m:
        dom = m.group(1)
        if any(td in dom and td != dom for td in TRUSTED_DOMAINS):
            score += 0.25
    for url in urls:
        try:
            netloc = urlparse(url).netloc.lower()
            if re.match(r'^\d+\.', netloc): score += 0.20
            if any(netloc.endswith(tld) for tld in SUSPICIOUS_TLDS): score += 0.15
        except:
            pass
    return min(1.0, score)

SUSPICIOUS_APIS = {'VirtualAllocEx','WriteProcessMemory','CreateRemoteThread',
                   'CryptEncrypt','RegSetValueEx','CryptAcquireContext'}

def quick_static_scan(binary_blob: bytes) -> Dict:
    raw = re.findall(rb'[\x20-\x7E]{4,}', binary_blob)
    strings = [s.decode('ascii','ignore') for s in raw]
    apis    = [s for s in strings if s in SUSPICIOUS_APIS]
    iocs    = [s for s in strings if 'http' in s.lower() or re.match(r'\d+\.\d+\.\d+', s)]
    risk    = min(100, len(apis)*15 + len(iocs)*10)
    return {'apis': apis, 'iocs': iocs, 'risk': risk,
            'sha256': hashlib.sha256(binary_blob).hexdigest()[:16]}

print('SOC pipeline components loaded')

In [ ]:
@dataclass
class SOCAlert:
    alert_id: str
    severity: str
    source:   str
    title:    str
    details:  Dict
    score:    float

class SOCPipeline:
    def __init__(self):
        self.alerts: List[SOCAlert] = []
        self._id = 0

    def _new_id(self) -> str:
        self._id += 1
        return f'ALERT-{self._id:04d}'

    def ingest_email(self, email: Dict) -> Optional[SOCAlert]:
        score = phishing_score_fast(email)
        if score >= 0.40:
            sev = 'Critical' if score >= 0.70 else 'High'
            alert = SOCAlert(
                alert_id=self._new_id(), severity=sev,
                source='Email Gateway',
                title=f'Phishing detected: {email.get("subject","")[:50]}',
                details={'from': email.get('from'), 'score': round(score,3)},
                score=score
            )
            self.alerts.append(alert)
            return alert
        return None

    def ingest_binary(self, name: str, data: bytes) -> Optional[SOCAlert]:
        result = quick_static_scan(data)
        if result['risk'] >= 30:
            sev = 'Critical' if result['risk'] >= 60 else 'High'
            alert = SOCAlert(
                alert_id=self._new_id(), severity=sev,
                source='Endpoint/EDR',
                title=f'Suspicious binary: {name}',
                details=result,
                score=result['risk']/100
            )
            self.alerts.append(alert)
            return alert
        return None

    def ingest_network_flow(self, flow: Dict) -> Optional[SOCAlert]:
        # Simple rules: large outbound after hours = exfil
        score = 0.0
        if flow.get('bytes_out', 0) > 1_000_000:
            score += 0.5
        if flow.get('hour_of_day', 12) < 6 or flow.get('hour_of_day', 12) > 22:
            score += 0.3
        if flow.get('unique_ports', 1) > 50:
            score += 0.4  # port scan
        score = min(1.0, score)
        if score >= 0.5:
            sev = 'Critical' if score >= 0.8 else 'High'
            alert = SOCAlert(
                alert_id=self._new_id(), severity=sev,
                source='Network IDS',
                title='Anomalous network flow detected',
                details={k:round(v,1) if isinstance(v,float) else v for k,v in flow.items()},
                score=score
            )
            self.alerts.append(alert)
            return alert
        return None

    def correlate(self) -> List[Dict]:
        """Group alerts by severity and find overlapping time windows."""
        critical = [a for a in self.alerts if a.severity == 'Critical']
        high     = [a for a in self.alerts if a.severity == 'High']
        multi_source = len({a.source for a in self.alerts}) > 1
        incidents = []
        if len(critical) >= 2 or (len(critical) >= 1 and multi_source):
            incidents.append({'severity': 'P1', 'alerts': [a.alert_id for a in critical],
                              'note': 'Possible active intrusion — escalate immediately'})
        elif high:
            incidents.append({'severity': 'P2', 'alerts': [a.alert_id for a in high],
                              'note': 'Investigate — possible targeted attack'})
        return incidents

soc = SOCPipeline()
print('SOC pipeline initialised')

In [ ]:
# --- Ingest events ---

# Email events
emails = [
    {'from': 'security@paypa1.com', 'subject': 'URGENT: Verify your account now!',
     'body': 'Click here immediately to verify or your account will be suspended.',
     'urls': ['http://203.0.113.9/paypal/verify']},
    {'from': 'newsletter@company.com', 'subject': 'Q1 Newsletter',
     'body': 'Here is our quarterly update.', 'urls': []},
    {'from': 'hr@corp-update.xyz', 'subject': 'Invoice FINAL NOTICE',
     'body': 'Pay invoice to avoid suspension. Expires today!',
     'urls': ['http://evil.top/invoice']},
]

# Binary samples
import struct
rng = random.Random(99)
mal_sample = (b'MZ' + b'\x90'*60 + b'PE\x00\x00' +
              b'VirtualAllocEx\x00WriteProcessMemory\x00CreateRemoteThread\x00' +
              b'http://c2.evil.example/beacon\x00' +
              bytes(rng.randint(0,255) for _ in range(100)))
clean_sample = b'MZ' + b'\x90'*60 + b'PE\x00\x00' + b'ReadFile\x00WriteFile\x00' + b'\x00'*100

# Network flows
flows = [
    {'bytes_out': 8000000, 'bytes_in': 2000, 'duration_sec': 400, 'packet_count': 5000, 'unique_ports': 1, 'hour_of_day': 2},
    {'bytes_out': 45000,   'bytes_in': 180000,'duration_sec': 25,  'packet_count': 120,  'unique_ports': 2, 'hour_of_day': 10},
    {'bytes_out': 300,     'bytes_in': 500,   'duration_sec': 3,   'packet_count': 600,  'unique_ports': 800,'hour_of_day': 3},
]

print('--- Ingesting Email Events ---')
for e in emails:
    alert = soc.ingest_email(e)
    if alert:
        print(f'  [{alert.severity}] {alert.alert_id}: {alert.title}')
    else:
        print(f'  [CLEAN] {e["subject"][:40]}')

print('\n--- Ingesting Binary Samples ---')
for name, sample in [('malware.exe', mal_sample), ('notepad.exe', clean_sample)]:
    alert = soc.ingest_binary(name, sample)
    if alert:
        print(f'  [{alert.severity}] {alert.alert_id}: {alert.title}')
    else:
        print(f'  [CLEAN] {name}')

print('\n--- Ingesting Network Flows ---')
for flow in flows:
    alert = soc.ingest_network_flow(flow)
    if alert:
        print(f'  [{alert.severity}] {alert.alert_id}: {alert.title} (bytes_out={flow["bytes_out"]})')
    else:
        print(f'  [CLEAN] flow hour={flow["hour_of_day"]} bytes_out={flow["bytes_out"]}')

In [ ]:
# --- Correlation and incident creation ---
print('\n=== SOC Dashboard ===')
print(f'Total alerts: {len(soc.alerts)}')
for a in soc.alerts:
    print(f'  [{a.severity:<8}] {a.alert_id} {a.source:<20} {a.title[:55]}')

incidents = soc.correlate()
print(f'\nCorrelated incidents: {len(incidents)}')
for inc in incidents:
    print(f'  [{inc["severity"]}] Alerts: {inc["alerts"]} — {inc["note"]}')

# --- Coverage metrics ---
sources  = {a.source for a in soc.alerts}
sevs     = {'Critical': 0, 'High': 0, 'Medium': 0}
for a in soc.alerts:
    sevs[a.severity] = sevs.get(a.severity, 0) + 1

print(f'\nAlert sources: {sources}')
print(f'By severity:   {sevs}')
print('\nSOC Platform capabilities demonstrated:')
capabilities = [
    'Static code vulnerability scanning (SAST)',
    'Coverage-guided fuzzing and payload mutation',
    'Network reconnaissance and attack surface scoring',
    'Credential stuffing detection',
    'NLP-based phishing classification',
    'Web application DAST (SQLi, XSS, IDOR, SSRF)',
    'Binary static analysis and YARA rule generation',
    'Threat intelligence attribution (Diamond Model)',
    'Red team operation planning with RoE enforcement',
    'ML anomaly detection (Z-Score + Isolation Forest)',
    'Multi-source alert correlation and incident creation',
]
for cap in capabilities:
    print(f'  [x] {cap}')